# Exploratory Data Analysis of DMEPOS Referring Provider

## Standardized visuals

In [1]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import seaborn as sns
import numpy as np

# =============================================================================
# 1. COLORBLIND-FRIENDLY PALETTE  (Wong 2011, Nature Methods)
#    These 8 colors are maximally distinguishable for all color vision types.
# =============================================================================
CB_PALETTE = [
    "#0072B2",  # blue
    "#E69F00",  # orange
    "#009E73",  # green
    "#CC79A7",  # pink
    "#56B4E9",  # sky blue
    "#D55E00",  # vermillion
    "#F0E442",  # yellow
    "#000000",  # black
]

# =============================================================================
# 2. FONT DETECTION & SETTINGS
#    Auto-picks the best sans-serif font already on your machine.
#    Minimum size is 18pt; titles and labels are larger.
#
#    >>> To see ALL fonts available on your system, run in a cell:
#    >>>     from plot_template import list_fonts
#    >>>     list_fonts()
#
#    >>> To override the auto-picked font, set FONT_FAMILY before importing:
#    >>>     import plot_template
#    >>>     plot_template.FONT_FAMILY = "Verdana"
#    >>>     plot_template.apply_style()   # re-apply with your choice
# =============================================================================

def list_fonts(filter_text=None):
    """
    Print every font matplotlib can see on your system.
    Pass a string to filter, e.g. list_fonts("arial") or list_fonts("sans").
    """
    names = sorted({f.name for f in fm.fontManager.ttflist})
    if filter_text:
        names = [n for n in names if filter_text.lower() in n.lower()]
    print(f"Found {len(names)} font{'s' if len(names) != 1 else ''}:\n")
    for n in names:
        print(f"  • {n}")
    return names

# Preferred fonts in priority order (all clean presentation sans-serifs)
_FONT_PREFERENCES = [
    "Arial",
    "Helvetica",
    "Helvetica Neue",
    "Calibri",
    "Verdana",
    "Segoe UI",
    "Liberation Sans",
    "DejaVu Sans",
]

def _find_best_font():
    """Return the first preferred font that is actually installed."""
    available = {f.name for f in fm.fontManager.ttflist}
    for font in _FONT_PREFERENCES:
        if font in available:
            return font
    # Nothing from the preferred list — just grab any sans-serif we can find
    for name in sorted(available):
        if "sans" in name.lower():
            return name
    # True last resort
    return "DejaVu Sans"

FONT_FAMILY = _find_best_font()
print(f"Font: {FONT_FAMILY}")
FONT_SIZE_TITLE   = 30      # figure / axes title
FONT_SIZE_LABEL   = 24      # axis labels
FONT_SIZE_TICK    = 20      # tick labels
FONT_SIZE_LEGEND  = 20      # legend text
FONT_SIZE_ANNOT   = 18      # annotations / text on plot

# =============================================================================
# 3. APPLY GLOBAL STYLE (matplotlib + seaborn)
#    Called automatically on import. Call again after changing FONT_FAMILY.
# =============================================================================

def apply_style():
    """Apply (or re-apply) the full presentation style using current settings."""
    mpl.rcParams.update({
        # --- Font -----------------------------------------------------------
        "font.family":        "sans-serif",
        "font.sans-serif":    [FONT_FAMILY, "Helvetica", "DejaVu Sans"],
        "font.size":          FONT_SIZE_ANNOT,       # base / fallback size

        # --- Axes -----------------------------------------------------------
        "axes.titlesize":     FONT_SIZE_TITLE,
        "axes.titleweight":   "bold",
        "axes.labelsize":     FONT_SIZE_LABEL,
        "axes.labelweight":   "bold",
        "axes.prop_cycle":    mpl.cycler(color=CB_PALETTE),
        "axes.linewidth":     1.5,
        "axes.edgecolor":     "#000000",
        "axes.spines.top":    False,
        "axes.spines.right":  False,
        "axes.grid":          True,
        "axes.axisbelow":     True,                   # grid behind data

        # --- Ticks ----------------------------------------------------------
        "xtick.labelsize":    FONT_SIZE_TICK,
        "ytick.labelsize":    FONT_SIZE_TICK,
        "xtick.major.width":  1.5,
        "ytick.major.width":  1.5,
        "xtick.major.size":   6,
        "ytick.major.size":   6,
        "xtick.direction":    "out",
        "ytick.direction":    "out",

        # --- Grid -----------------------------------------------------------
        "grid.color":         "#dbdbdb",
        "grid.linewidth":     0.8,
        "grid.alpha":         1.0,

        # --- Legend ----------------------------------------------------------
        "legend.fontsize":    FONT_SIZE_LEGEND,
        "legend.frameon":     True,
        "legend.framealpha":  0.9,
        "legend.edgecolor":   "#999999",
        "legend.fancybox":    True,
        "legend.shadow":      False,
        "legend.loc":         "best",

        # --- Lines & markers ------------------------------------------------
        "lines.linewidth":    2.5,
        "lines.markersize":   10,

        # --- Figure ---------------------------------------------------------
        "figure.figsize":     (12, 7),
        "figure.dpi":         100,
        "figure.facecolor":   "white",
        "figure.titlesize":   FONT_SIZE_TITLE,
        "figure.titleweight": "bold",

        # --- Saving ---------------------------------------------------------
        "savefig.dpi":        300,
        "savefig.bbox":       "tight",
        "savefig.facecolor":  "white",
    })

    sns.set_theme(
        style="whitegrid",
        font=FONT_FAMILY,
        font_scale=1.4,
        rc=mpl.rcParams,
    )
    sns.set_palette(CB_PALETTE)

    # Re-assert settings that seaborn's set_theme can override
    mpl.rcParams.update({
        "axes.titlesize":     FONT_SIZE_TITLE,
        "axes.titleweight":   "bold",
        "axes.labelweight":   "bold",
        "axes.edgecolor":     "#000000",
        "axes.spines.top":    False,
        "axes.spines.right":  False,
        "figure.titlesize":   FONT_SIZE_TITLE,
        "figure.titleweight": "bold",
        "grid.color":         "#dbdbdb",
        "grid.alpha":         1.0,
    })

# Auto-apply on import
apply_style()

# =============================================================================
# 5. HELPER FUNCTIONS
# =============================================================================

def add_legend(ax=None, title=None, **kwargs):
    """
    Add a styled legend to the current or specified axes.
    Merges template defaults with any overrides you pass in.
    """
    ax = ax or plt.gca()
    defaults = dict(
        fontsize=FONT_SIZE_LEGEND,
        frameon=True,
        framealpha=0.9,
        edgecolor="#999999",
        loc="best",
    )
    defaults.update(kwargs)
    legend = ax.legend(title=title, **defaults)
    if title:
        legend.get_title().set_fontsize(FONT_SIZE_LEGEND)
        legend.get_title().set_fontweight("bold")
    return legend


def get_colors(n=None):
    """
    Return the colorblind-friendly palette (or the first n colors).
    Useful when you need explicit color assignments.

    Usage:
        colors = get_colors(3)
        ax.bar(x, y1, color=colors[0], label="Group A")
        ax.bar(x, y2, color=colors[1], label="Group B")
    """
    if n is None:
        return CB_PALETTE.copy()
    return CB_PALETTE[:n]


def finalize(fig=None, tight=True):
    """
    Call at the end of every plot cell to tighten layout.
    """
    fig = fig or plt.gcf()
    if tight:
        fig.tight_layout()

Font: Liberation Sans


## Read-in the dataset

In [1]:
import pandas as pd

pd.set_option('display.max_rows',None)
pd.set_option('display.max_columns',None)

# read-in the dataset
df = pd.read_csv('/dsa/groups/casestudycf25/team02/silver/dmepos_rfrr_labeled.csv',dtype={'rfrg_prvdr_state_fips':str,'rfrg_prvdr_zip5':str})
df.head()

,npi,rfrg_prvdr_last_name_org,rfrg_prvdr_first_name,rfrg_prvdr_mi,rfrg_prvdr_crdntls,rfrg_prvdr_ent_cd,rfrg_prvdr_st1,rfrg_prvdr_st2,rfrg_prvdr_city,rfrg_prvdr_state_abrvtn,rfrg_prvdr_state_fips,rfrg_prvdr_zip5,rfrg_prvdr_ruca,rfrg_prvdr_ruca_desc,rfrg_prvdr_cntry,rfrg_prvdr_spclty_desc,rfrg_prvdr_spclty_srce,tot_suplrs,tot_suplr_hcpcs_cds,tot_suplr_benes,tot_suplr_clms,tot_suplr_srvcs,suplr_sbmtd_chrgs,suplr_mdcr_alowd_amt,suplr_mdcr_pymt_amt,suplr_mdcr_stdzd_pymt_amt,dme_sprsn_ind,dme_tot_suplrs,dme_tot_suplr_hcpcs_cds,dme_tot_suplr_benes,dme_tot_suplr_clms,dme_tot_suplr_srvcs,dme_suplr_sbmtd_chrgs,dme_suplr_mdcr_alowd_amt,dme_suplr_mdcr_pymt_amt,dme_suplr_mdcr_stdzd_pymt_amt,pos_sprsn_ind,pos_tot_suplrs,pos_tot_suplr_hcpcs_cds,pos_tot_suplr_benes,pos_tot_suplr_clms,pos_tot_suplr_srvcs,pos_suplr_sbmtd_chrgs,pos_suplr_mdcr_alowd_amt,pos_suplr_mdcr_pymt_amt,pos_suplr_mdcr_stdzd_pymt_amt,drug_sprsn_ind,drug_tot_suplrs,drug_tot_suplr_hcpcs_cds,drug_tot_suplr_benes,drug_tot_suplr_clms,drug_tot_suplr_srvcs,drug_suplr_sbmtd_chrgs,drug_suplr_mdcr_alowd_amt,drug_suplr_mdcr_pymt_amt,drug_suplr_mdcr_stdzd_pymt_amt,bene_avg_age,bene_age_lt_65_cnt,bene_age_65_74_cnt,bene_age_75_84_cnt,bene_age_gt_84_cnt,bene_feml_cnt,bene_male_cnt,bene_race_wht_cnt,bene_race_black_cnt,bene_race_api_cnt,bene_race_hspnc_cnt,bene_race_natind_cnt,bene_race_othr_cnt,bene_ndual_cnt,bene_dual_cnt,bene_cc_bh_adhd_oth_cd_v1_pct,bene_cc_bh_alcohol_drug_v1_pct,bene_cc_bh_tobacco_v1_pct,bene_cc_bh_alz_non_alzdem_v2_pct,bene_cc_bh_anxiety_v1_pct,bene_cc_bh_bipolar_v1_pct,bene_cc_bh_mood_v2_pct,bene_cc_bh_depress_v1_pct,bene_cc_bh_pd_v1_pct,bene_cc_bh_ptsd_v1_pct,bene_cc_bh_schizo_oth_psy_v1_pct,bene_cc_ph_asthma_v2_pct,bene_cc_ph_afib_v2_pct,bene_cc_ph_cancer6_v2_pct,bene_cc_ph_ckd_v2_pct,bene_cc_ph_copd_v2_pct,bene_cc_ph_diabetes_v2_pct,bene_cc_ph_hf_non_ihd_v2_pct,bene_cc_ph_hyperlipidemia_v2_pct,bene_cc_ph_hypertension_v2_pct,bene_cc_ph_ischemic_heart_v2_pct,bene_cc_ph_osteoporosis_v2_pct,bene_cc_ph_parkinson_v2_pct,bene_cc_ph_arthritis_v2_pct,bene_cc_ph_stroke_tia_v2_pct,bene_avg_risk_scre,year,target
0,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,24,20817,1.0,Metropolitan area core: primary flow within an...,US,Internal Medicine,Claim-Specialty,9,16,12.0,38,95,16659.18,4554.39,3515.05,3827.20,NaN,9.0,16.0,12.0,38.0,95.0,16659.180,4554.39,3515.05,3827.200,NaN,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,79.750000,0.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,0.0,0.0,-1.0,-1.0,0.0,-1.0,-1.0,-1.0,-1.0,0.0,-1.0,-1.0,0.0,0.0,0.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.000000,-1.0,-1.000000,1.000000,-1.0,-1.0,0.0,-1.0,-1.0,1.942167,2021,0
1,1003000480,Rothchild,Kevin,B,md,I,12605 E 16th Ave,NaN,Aurora,CO,08,80045,1.0,Metropolitan area core: primary flow within an...,US,General Surgery,Claim-Specialty,4,3,5.0,11,24,4863.99,1353.37,1082.62,1360.53,NaN,4.0,3.0,5.0,11.0,24.0,4863.990,1353.37,1082.62,1360.530,NaN,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,68.200000,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.000000,-1.0,-1.000000,-1.000000,-1.0,-1.0,-1.0,-1.0,-1.0,2.278200,2021,0
2,1003000522,Weigand,Frederick,J,md,I,1565 Saxon Blvd,Suite 102,Deltona,FL,12,32725,1.1,Secondary flow 30% to <50% to a larger urbaniz...,US,Family Practice,NPPES-Specialty,10,22,13.0,47,201,11438.01,2469.17,1846.10,2103.95,NaN,9.0,16.0,12.0,44.0,135.0,10671.850,1911.33,1405.54,1603.420,*,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,*,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,79.923077,0.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,0.0,0.0,0.0,13.0,0.0,0.0,-1.0,-1.0,-1.0,-1.0,0.0,-1.0,-1.0,0.0,0.0,0.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.000000,-1.0,1.000000,0.846154,-1.0,-1.0,0.0,-1.0,-1.0,1.872308,2021,0
3,1003000530,Semonche,Amanda,M,do,I,1021 Park Ave,Suite 203,Quakertown,PA,42,18951,1.0,Metropolitan area core: pri

## Analyze the class imbalance

In [2]:
# all providers
df_imbal_nat = df.filter(items=['npi','rfrg_prvdr_state_abrvtn','target'])
# MO and bordering states
df_imbal_brd = df_imbal_nat[df_imbal_nat.rfrg_prvdr_state_abrvtn.isin(['MO', 'IL', 'IA', 'OK', 'AR', 'TN', 'KS', 'NE', 'KY'])]
# MO only
df_imbal_mo = df_imbal_nat[df_imbal_nat.rfrg_prvdr_state_abrvtn == 'MO']

In [3]:
pvdr_srvcs_all = df_imbal_nat.shape[0]
frd_all = df_imbal_nat['target'].sum()

pvdr_srvcs_brd = df_imbal_brd.shape[0]
frd_brd = df_imbal_brd['target'].sum()

pvdr_srvcs_mo = df_imbal_mo.shape[0]
frd_mo = df_imbal_mo['target'].sum()

print(f'MO amounts: {frd_mo} {pvdr_srvcs_mo} {frd_mo/pvdr_srvcs_mo}')
print(f'MO & Bordering amounts: {frd_brd} {pvdr_srvcs_brd} {frd_brd/pvdr_srvcs_brd}')
print(f'National amounts: {frd_all} {pvdr_srvcs_all} {frd_all/pvdr_srvcs_all}')

MO amounts: 23 23649 0.0009725569791534526
MO & Bordering amounts: 72 179322 0.0004015123632348513
National amounts: 321 1157253 0.00027738100484509437


## Wilcoxon rank-sum test on numeric features
All features right of and including `tot_suplrs` except `sprsn_ind`s, `year`, and `target`.

In [3]:
df_num = df.loc[:,'tot_suplrs':'target']
df_num = df_num.drop(columns=['dme_sprsn_ind','pos_sprsn_ind','drug_sprsn_ind','year'])
df_num.head()

,tot_suplrs,tot_suplr_hcpcs_cds,tot_suplr_benes,tot_suplr_clms,tot_suplr_srvcs,suplr_sbmtd_chrgs,suplr_mdcr_alowd_amt,suplr_mdcr_pymt_amt,suplr_mdcr_stdzd_pymt_amt,dme_tot_suplrs,dme_tot_suplr_hcpcs_cds,dme_tot_suplr_benes,dme_tot_suplr_clms,dme_tot_suplr_srvcs,dme_suplr_sbmtd_chrgs,dme_suplr_mdcr_alowd_amt,dme_suplr_mdcr_pymt_amt,dme_suplr_mdcr_stdzd_pymt_amt,pos_tot_suplrs,pos_tot_suplr_hcpcs_cds,pos_tot_suplr_benes,pos_tot_suplr_clms,pos_tot_suplr_srvcs,pos_suplr_sbmtd_chrgs,pos_suplr_mdcr_alowd_amt,pos_suplr_mdcr_pymt_amt,pos_suplr_mdcr_stdzd_pymt_amt,drug_tot_suplrs,drug_tot_suplr_hcpcs_cds,drug_tot_suplr_benes,drug_tot_suplr_clms,drug_tot_suplr_srvcs,drug_suplr_sbmtd_chrgs,drug_suplr_mdcr_alowd_amt,drug_suplr_mdcr_pymt_amt,drug_suplr_mdcr_stdzd_pymt_amt,bene_avg_age,bene_age_lt_65_cnt,bene_age_65_74_cnt,bene_age_75_84_cnt,bene_age_gt_84_cnt,bene_feml_cnt,bene_male_cnt,bene_race_wht_cnt,bene_race_black_cnt,bene_race_api_cnt,bene_race_hspnc_cnt,bene_race_natind_cnt,bene_race_othr_cnt,bene_ndual_cnt,bene_dual_cnt,bene_cc_bh_adhd_oth_cd_v1_pct,bene_cc_bh_alcohol_drug_v1_pct,bene_cc_bh_tobacco_v1_pct,bene_cc_bh_alz_non_alzdem_v2_pct,bene_cc_bh_anxiety_v1_pct,bene_cc_bh_bipolar_v1_pct,bene_cc_bh_mood_v2_pct,bene_cc_bh_depress_v1_pct,bene_cc_bh_pd_v1_pct,bene_cc_bh_ptsd_v1_pct,bene_cc_bh_schizo_oth_psy_v1_pct,bene_cc_ph_asthma_v2_pct,bene_cc_ph_afib_v2_pct,bene_cc_ph_cancer6_v2_pct,bene_cc_ph_ckd_v2_pct,bene_cc_ph_copd_v2_pct,bene_cc_ph_diabetes_v2_pct,bene_cc_ph_hf_non_ihd_v2_pct,bene_cc_ph_hyperlipidemia_v2_pct,bene_cc_ph_hypertension_v2_pct,bene_cc_ph_ischemic_heart_v2_pct,bene_cc_ph_osteoporosis_v2_pct,bene_cc_ph_parkinson_v2_pct,bene_cc_ph_arthritis_v2_pct,bene_cc_ph_stroke_tia_v2_pct,bene_avg_risk_scre,target
0,9,16,12.0,38,95,16659.18,4554.39,3515.05,3827.20,9.0,16.0,12.0,38.0,95.0,16659.180,4554.39,3515.05,3827.200,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,79.750000,0.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,0.0,0.0,-1.0,-1.0,0.0,-1.0,-1.0,-1.0,-1.0,0.0,-1.0,-1.0,0.0,0.0,0.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.000000,-1.0,-1.000000,1.000000,-1.0,-1.0,0.0,-1.0,-1.0,1.942167,0
1,4,3,5.0,11,24,4863.99,1353.37,1082.62,1360.53,4.0,3.0,5.0,11.0,24.0,4863.990,1353.37,1082.62,1360.530,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,68.200000,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.000000,-1.0,-1.000000,-1.000000,-1.0,-1.0,-1.0,-1.0,-1.0,2.278200,0
2,10,22,13.0,47,201,11438.01,2469.17,1846.10,2103.95,9.0,16.0,12.0,44.0,135.0,10671.850,1911.33,1405.54,1603.420,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,79.923077,0.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,0.0,0.0,0.0,13.0,0.0,0.0,-1.0,-1.0,-1.0,-1.0,0.0,-1.0,-1.0,0.0,0.0,0.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.000000,-1.0,1.000000,0.846154,-1.0,-1.0,0.0,-1.0,-1.0,1.872308,0
3,14,35,18.0,66,654,19129.66,7237.17,5619.54,6297.98,12.0,23.0,15.0,57.0,217.0,14328.450,4414.93,3476.29,4125.560,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,74.666667,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,18.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,0.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.722222,-1.0,0.833333,0.777778,-1.0,-1.0,-1.0,-1.0,-1.0,2.144684,0
4,4,11,5.0,14,2274,13075.14,7173.71,5449.53,5994.17,3.0,6.0,5.0,19.0,43.0,6075.045,1947.45,1449.27,1540.015,4.0,9.0,0.0,14.0,1434.0,12722.04,7042.01,5356.65,5903.16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,70.250000,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.000000,-1.0,-1.000000,-1.000000,-1.0,-1.0,-1.0,-1.0,-1.0,2.006500,0


In [4]:
from scipy import stats

feats = df_num.columns[:-1]

results = {}

for feat in feats:
    stat, pval = stats.ranksums(df_num[df_num['target']==0][feat], df_num[df_num['target']==1][feat])
    results[feat] = (stat, pval)

results

{'tot_suplrs': (-3.6371224048526725, 0.0002757009319622294),
 'tot_suplr_hcpcs_cds': (-2.940451074877442, 0.0032773475400156935),
 'tot_suplr_benes': (-1.352695451203889, 0.17615294424558903),
 'tot_suplr_clms': (-1.1039937764078436, 0.26959583741854676),
 'tot_suplr_srvcs': (-2.7973890818074, 0.005151745260183874),
 'suplr_sbmtd_chrgs': (-1.1690265198748004, 0.2423929467331879),
 'suplr_mdcr_alowd_amt': (-1.701305197797997, 0.0888856924559755),
 'suplr_mdcr_pymt_amt': (-1.5612043387920644, 0.11847554561434184),
 'suplr_mdcr_stdzd_pymt_amt': (-1.4652047600997273, 0.1428650612932944),
 'dme_tot_suplrs': (-3.012067374665086, 0.002594750002956374),
 'dme_tot_suplr_hcpcs_cds': (-3.0436360949363963, 0.0023373769106981883),
 'dme_tot_suplr_benes': (-1.7474278560684533, 0.08056314900853169),
 'dme_tot_suplr_clms': (-1.6034896039254358, 0.10882660253928612),
 'dme_tot_suplr_srvcs': (-2.005884429852233, 0.04486857983391333),
 'dme_suplr_sbmtd_chrgs': (-1.0950450391091957, 0.2734969052511208),
 

Our test shows that the most significant differences between the fraud and non-fraud classes are in `tot_suplrs`, `bene_ndual_cnt`, `bene_dual_cnt`, and `bene_avg_risk_scre` for which the fraud class values are lower with p-values 0.00028, 0.00042, 0.00004, and 0.00047 respectively.

In [5]:
for key in results:
    if results[key][1] < 0.0005:
        print(f'{key}: {results[key]}')


tot_suplrs: (-3.6371224048526725, 0.0002757009319622294)
bene_ndual_cnt: (-3.5244320458604252, 0.0004243916534164177)
bene_dual_cnt: (-4.108111573028068, 3.989073980735861e-05)
bene_avg_risk_scre: (-3.4974302662305115, 0.00046976350784725415)


### Are the differences maintained after filtering for MO and bordering states?

In [7]:
df_brd = df[df.rfrg_prvdr_state_abrvtn.isin(['MO', 'IL', 'IA', 'OK', 'AR', 'TN', 'KS', 'NE', 'KY'])]
df_brd_num = df_brd.loc[:,'tot_suplrs':'target']
df_brd_num = df_brd_num.drop(columns=['dme_sprsn_ind','pos_sprsn_ind','drug_sprsn_ind','year'])

feats = df_num.columns[:-1]

results = {}

for feat in feats:
    stat, pval = stats.ranksums(df_brd_num[df_brd_num['target']==0][feat], df_brd_num[df_brd_num['target']==1][feat])
    results[feat] = (stat, pval)

results

{'tot_suplrs': (-1.864414317018941, 0.06226354386665823),
 'tot_suplr_hcpcs_cds': (-0.2358887369124232, 0.8135190083043905),
 'tot_suplr_benes': (-1.1577090307358047, 0.24698279737608841),
 'tot_suplr_clms': (-1.3123985127378714, 0.18938570844384295),
 'tot_suplr_srvcs': (-0.6421036103921599, 0.5208059118797876),
 'suplr_sbmtd_chrgs': (-1.2116597015694135, 0.2256426710369791),
 'suplr_mdcr_alowd_amt': (-1.6329590396140952, 0.10247761165245975),
 'suplr_mdcr_pymt_amt': (-1.5521912898792323, 0.12061645958902464),
 'suplr_mdcr_stdzd_pymt_amt': (-1.4347774792377437, 0.15135051740392647),
 'dme_tot_suplrs': (-2.076908414495898, 0.037810012988731524),
 'dme_tot_suplr_hcpcs_cds': (-0.999137140690379, 0.31772826140104704),
 'dme_tot_suplr_benes': (-2.296718744842833, 0.021634819423045692),
 'dme_tot_suplr_clms': (-2.4399716318901956, 0.01468841530466944),
 'dme_tot_suplr_srvcs': (-2.0002792434230905, 0.04547011907066786),
 'dme_suplr_sbmtd_chrgs': (-2.1040294825313386, 0.035375875308699956),
 

In [8]:
for key in results:
    if results[key][1] < 0.0005:
        print(f'{key}: {results[key]}')

While the values remain generally lower for the fraud class within MO and bordering states, the differences are shown to be far less significant in this region. In fact, no fields display differences between the fraud and non-fraud classes with extremely high significance (p-value = 0.0005) within just MO and bordering states.